# DFTracer Notebook Workspace + MCP Agent

Use this notebook as the **main user interface** for the project.

## What this notebook does
- asks for a GitHub repository URL first
- creates a clean per-repository workspace under `workspaces/`
- clones or updates the source code
- creates a dedicated virtual environment for that workspace
- installs this project and notebook dependencies into that workspace venv
- starts the local MCP service directly from the notebook
- runs the guided DFTracer pipeline (detect -> decide features -> install dftracer -> annotate -> run -> postprocess)

## Recommended flow
1. Run the next cell and enter the GitHub URL.
2. Prepare the workspace.
3. Install notebook/project dependencies (no DFTracer build yet).
4. Start the MCP agent.
5. Review detected repo settings and answer the guided questions.
6. Run the full guided pipeline.
7. Ask follow-up questions in the notebook chat section.

## 1. App configuration

Start here.

Enter the repository URL, choose a **branch or tag**, and prepare a workspace with this layout:
- `source/` cloned application repo
- `external/` third-party source trees such as `dftracer`
- `build/` build directories
- `install/` local install prefix for tools
- `venv/` per-workspace Python environment
- `traces/` trace outputs
- `artifacts/` saved pipeline outputs
- `logs/` install/build logs

In [1]:
from IPython import get_ipython

_ip = get_ipython()
if _ip is not None:
    try:
        _ip.run_line_magic("load_ext", "autoreload")
    except Exception:
        pass
    _ip.run_line_magic("autoreload", "2")

import json
import pathlib
import subprocess
from pprint import pprint

from dftracer_agents.workspace import workspace_env
from dftracer_agents.notebook.config import install_notebook_config
from dftracer_agents.notebook.session import install_notebook_session
from dftracer_agents.notebook.widgets import install_notebook_widgets

CONFIG_RUNTIME = install_notebook_config(globals())
SESSION_RUNTIME = install_notebook_session(globals())
WIDGET_RUNTIME = install_notebook_widgets(globals())

DEFAULT_REPO_URL = globals()["DEFAULT_REPO_URL"]
DEFAULT_REPO_REF = globals()["DEFAULT_REPO_REF"]
DEFAULT_COMPILER_MODULE = globals()["DEFAULT_COMPILER_MODULE"]
PROJECT_ROOT = globals()["PROJECT_ROOT"]
WORKSPACES_ROOT = globals()["WORKSPACES_ROOT"]
APP_STATE = globals()["APP_STATE"]
USE_WIDGETS = globals()["USE_WIDGETS"]
USE_WIDGETS_MCP = globals()["USE_WIDGETS_MCP"]
effective_config = globals()["effective_config"]
update_latest_agent_code = globals()["update_latest_agent_code"]
prepare_workspace = globals()["prepare_workspace"]
install_workspace_deps = globals()["install_workspace_deps"]
show_agent_env = globals()["show_agent_env"]
start_local_agent = globals()["start_local_agent"]
stop_local_agent = globals()["stop_local_agent"]
ask_agent = globals()["ask_agent"]
render_workspace_setup_section = globals()["render_workspace_setup_section"]
render_install_section = globals()["render_install_section"]
render_agent_controls_section = globals()["render_agent_controls_section"]
render_feedback_section = globals()["render_feedback_section"]
render_pipeline_run_section = globals()["render_pipeline_run_section"]
render_chat_section = globals()["render_chat_section"]
render_outcome_feedback_section = globals()["render_outcome_feedback_section"]
run_last_failed_stage = globals().get("run_last_failed_stage")

print(f"Widget UI enabled for non-MCP sections: {USE_WIDGETS}")
print(f"Widget UI enabled for MCP controls: {USE_WIDGETS_MCP}")
print("Notebook autoreload enabled: source edits will be picked up after rerunning this cell.")
print("run_last_failed_stage becomes available after the pipeline setup cell in section 6 runs.")

update_latest_agent_code()
render_workspace_setup_section()

Widget UI enabled for non-MCP sections: True
Widget UI enabled for MCP controls: False
Notebook autoreload enabled: source edits will be picked up after rerunning this cell.
run_last_failed_stage becomes available after the pipeline setup cell in section 6 runs.
$ /usr/workspace/haridev/dftracer-agents/.venv/bin/python -m pip install -e /usr/workspace/haridev/dftracer-agents
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
Obtaining file:///usr/workspace/haridev/dftracer-agents
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'don

## 2. Install Notebook and Project Dependencies

This step installs only what the notebook and project need into the **workspace venv**.

It intentionally does **not** install/build DFTracer yet.
DFTracer installation is done later in the pipeline after the app is detected and
feature flags are determined (`install_dftracer` stage).

If section 1 has not been run, this cell auto-prepares the workspace from the current
GitHub URL / selected branch-or-tag fields.

In [2]:
render_install_section()

## 3. Start the Local MCP Service and Goose Recipe Runner

The notebook now centralizes both runtime paths:

- the notebook-local MCP-backed Python runtime
- Goose CLI for one-shot prompts
- the numbered Goose YAML main recipe and stage subrecipes that keep MCP as the tool backend

Controls below let you:
- load `.env`
- map `LIVAI_*` variables to both `OPENAI_*` and Goose-compatible settings
- start or stop the local MCP-backed runtime when needed
- inspect the effective endpoint and model configuration
- run Goose with a default prompt aimed at the main recipe
- run any numbered Goose pipeline recipe stage directly from the notebook

In [3]:
render_agent_controls_section()

MCP widget controls disabled for stability. Run manually:
  show_agent_env()
  await start_local_agent()
  await stop_local_agent()


In [4]:
import json
import os
from pprint import pprint

import ipywidgets as widgets
from IPython.display import display

load_project_env()

def goose_env_snapshot() -> dict[str, str]:
    load_project_env()
    keys = [
        "LIVAI_BASE_URL",
        "LIVAI_MODEL",
        "LIVAI_API_KEY",
        "OPENAI_BASE_URL",
        "OPENAI_HOST",
        "OPENAI_BASE_PATH",
        "OPENAI_MODEL",
        "GOOSE_PROVIDER",
        "GOOSE_MODEL",
        "GOOSE_EDITOR_HOST",
        "GOOSE_EDITOR_MODEL",
    ]
    snapshot: dict[str, str] = {}
    for key in keys:
        value = os.environ.get(key, "")
        if key.endswith("API_KEY") and value:
            value = f"{'*' * max(0, len(value) - 4)}{value[-4:]}"
        snapshot[key] = value
    return snapshot

def default_main_recipe_prompt(stage_name: str = "detect") -> str:
    recipe_path = SESSION_RUNTIME.goose_pipeline_recipe_path()
    return (
        f"Use the DFTracer Goose main recipe at {recipe_path}. "
        f"Run the stage named '{stage_name}' for the prepared workspace, rely on the attached DFTracer MCP tools, "
        "and return a short execution summary followed by the structured result payload."
    )

goose_prompt = widgets.Textarea(
    value=default_main_recipe_prompt(),
    placeholder="Ask Goose to inspect, plan, or execute a DFTracer workflow task...",
    description="Prompt",
    layout=widgets.Layout(width="100%", height="140px"),
)
goose_stage = widgets.Dropdown(
    options=[
        ("detect", "detect"),
        ("test_default_build_setup", "test_default_build_setup"),
        ("test_default_run", "test_default_run"),
        ("annotate", "annotate"),
        ("build_with_dftracer", "build_with_dftracer"),
        ("postprocess", "postprocess"),
        ("dfanalyzer", "dfanalyzer"),
    ],
    value="detect",
    description="Recipe",
)
goose_context = widgets.Textarea(
    value="",
    placeholder="Optional extra context passed into the numbered Goose pipeline recipe...",
    description="Context",
    layout=widgets.Layout(width="100%", height="120px"),
)
goose_run_button = widgets.Button(description="Run Goose Prompt", button_style="primary", icon="play")
goose_main_prompt_button = widgets.Button(description="Run Main Recipe Prompt", button_style="success", icon="rocket")
goose_reset_prompt_button = widgets.Button(description="Reset Prompt", button_style="", icon="refresh")
goose_recipe_button = widgets.Button(description="Run Pipeline Recipe", button_style="warning", icon="cogs")
goose_env_button = widgets.Button(description="Show Goose Env", button_style="info", icon="search")
goose_output = widgets.Output()

def _on_goose_env(_button) -> None:
    with goose_output:
        goose_output.clear_output()
        pprint(goose_env_snapshot())
        print("\nRecipe file:")
        print(SESSION_RUNTIME.goose_pipeline_recipe_path())

def _on_goose_run(_button) -> None:
    with goose_output:
        goose_output.clear_output()
        prompt = goose_prompt.value.strip()
        if not prompt:
            print("Enter a Goose prompt first.")
            return
        result = SESSION_RUNTIME.run_goose_prompt(prompt)
        print(result)

def _on_goose_main_prompt(_button) -> None:
    goose_prompt.value = default_main_recipe_prompt(goose_stage.value)
    _on_goose_run(_button)

def _on_goose_reset_prompt(_button) -> None:
    goose_prompt.value = default_main_recipe_prompt(goose_stage.value)

def _on_goose_recipe(_button) -> None:
    with goose_output:
        goose_output.clear_output()
        stage_name = goose_stage.value
        context = goose_context.value.strip()
        print(f"Running recipe stage: {stage_name}")
        payload = SESSION_RUNTIME.run_goose_pipeline_stage_recipe(stage_name, context)
        print(json.dumps(payload, indent=2, sort_keys=True))

goose_env_button.on_click(_on_goose_env)
goose_run_button.on_click(_on_goose_run)
goose_main_prompt_button.on_click(_on_goose_main_prompt)
goose_reset_prompt_button.on_click(_on_goose_reset_prompt)
goose_recipe_button.on_click(_on_goose_recipe)

display(
    widgets.VBox(
        [
            widgets.HTML(
                "<b>Goose from the notebook</b><br>"
                "Uses the same .env/LIVAI settings and launches Goose with the local DFTracer MCP server attached. "
                "The default prompt targets the numbered main recipe, and the interface below can also run individual recipe stages directly."
            ),
            goose_prompt,
            widgets.HBox([goose_run_button, goose_main_prompt_button, goose_reset_prompt_button, goose_env_button]),
            widgets.HTML("<hr><b>Numbered Goose Pipeline Recipe Interface</b>"),
            goose_stage,
            goose_context,
            goose_recipe_button,
            goose_output,
        ]
    )
)

In [7]:
await stop_local_agent()

Agent stopped.


In [8]:
await start_local_agent()

Agent endpoint diagnostics:
  provider guess : livai
  api mode       : chat_completions
  host           : livai-api.llnl.gov
  path           : /v1
  api-version    : <unset>
  model          : gpt-5.4
  api key        : *********************lK7Q
✓ Agent started with model=gpt-5.4
✓ MCP tools loaded: 12
  • detect_available_modules
  • resolve_cmake_package_variables
  • detect_dftracer_profile
  • generate_annotation_plan
  • generate_cpp_compile_instructions
  • generate_runtime_env
  • auto_annotate_application
  • annotate_and_create_patch
  • generate_postprocess_plan
  • generate_layered_analysis_plan
  • build_end_to_end_pipeline
  • execute_pipeline_stage


## 4. User-Centric Questionnaire

This section asks for the missing details using **options**, so the pipeline stays guided and user-friendly.

The notebook combines:
- auto-detected repo facts
- your selected options
- your later follow-up feedback

Your choices here override detection when needed.

In [9]:
render_feedback_section()


## 5. Repository Summary and Effective Configuration

This section shows what was detected from the cloned repo and merges it with your answers from section 4.

In [10]:
layout = APP_STATE.get("workspace")
if not layout:
    raise RuntimeError("Prepare the workspace first in section 1.")

print(f"Repository: {layout.repo}")
print("\nDetected attributes:")
pprint(APP_STATE.get("repo_attrs") or {})
print("\nSource tree sample:")
for line in APP_STATE.get("tree_summary", [])[:80]:
    print(f"  {line}")

Repository: /usr/WS2/haridev/dftracer-agents/workspaces/ior/source/ior

Detected attributes:
{'has_cmake': False,
 'has_cpp': True,
 'has_pyproject': False,
 'has_python': True,
 'language': 'cpp',
 'uses_hip': False,
 'uses_mpi': True}

Source tree sample:
  AUTHORS
  COPYRIGHT
  META
  Makefile.am
  Makefile.in
  NEWS
  README.md
  README_DAOS
  README_S3
  aclocal.m4
  autom4te.cache/
  autom4te.cache/output.0
  autom4te.cache/output.1
  autom4te.cache/requests
  autom4te.cache/traces.0
  autom4te.cache/traces.1
  bootstrap
  config/
  config/ax_prog_cc_mpi.m4
  config/compile
  config/config.guess
  config/config.sub
  config/depcomp
  config/install-sh
  config/missing
  config/test-driver
  config/x_ac_meta.m4
  configure
  configure.ac
  contrib/
  contrib/Makefile.am
  contrib/Makefile.in
  contrib/cbif.c
  doc/
  doc/Makefile.am
  doc/Makefile.in
  doc/USER_GUIDE
  doc/doxygen/
  doc/doxygen/Doxyfile
  doc/image-sources/
  doc/image-sources/ior-accesses.pptx
  doc/mdtest.1
  d

In [11]:
CONFIG = effective_config()
pprint(CONFIG)


{'artifact_dir': '/usr/WS2/haridev/dftracer-agents/workspaces/ior/artifacts',
 'branch': '4.0.0',
 'build_system': 'auto',
 'detail_level': 'detailed',
 'goals': [],
 'install_dir': '/usr/WS2/haridev/dftracer-agents/workspaces/ior/install',
 'language': 'cpp',
 'notes': '',
 'repo_dir': '/usr/WS2/haridev/dftracer-agents/workspaces/ior/source/ior',
 'repo_url': 'https://github.com/hpc/ior',
 'trace_dir': '/usr/WS2/haridev/dftracer-agents/workspaces/ior/traces',
 'uses_hip': False,
 'uses_mpi': False,
 'venv_dir': '/usr/WS2/haridev/dftracer-agents/workspaces/ior/venv',
 'workload_type': 'general'}


## 6. Define the Guided Pipeline

The pipeline now runs **9 stages in order** and writes a `pipeline_state.json` file into each run artifact directory so failed stages can be rerun without discarding prior successful stages.

| # | Stage | What it does |
|---|-------|-------------|
| 1 | `detect` | Detects app language, build system, I/O patterns and DFTracer feature flags |
| 2 | `test_default_build_setup` | Builds and installs the app into the **workspace venv prefix** (not `app/install`) |
| 3 | `test_default_run` | Runs a baseline smoke test (without DFTracer instrumentation) |
| 4 | `install_dftracer` | Installs `dftracer[dfanalyzer]` with detected flags into the workspace venv |
| 5 | `annotate` | Automatically modifies source using DFTracer C/C++ and Python annotation patterns |
| 6 | `build_with_dftracer` | Rebuilds/reinstalls the app so annotated code is linked against DFTracer |
| 7 | `run_with_dftracer` | Runs the updated app with DFTracer runtime environment enabled |
| 8 | `postprocess` | Runs `dftracer-split` to compact traces and build the index |
| 9 | `dfanalyzer` | Runs the documented DFAnalyzer Python API flow on the compacted trace output |

In [12]:
from dftracer_agents.notebook.pipeline import install_notebook_pipeline

PIPELINE_RUNTIME = install_notebook_pipeline(globals())

PIPELINE_RESULTS = globals()["PIPELINE_RESULTS"]
PIPELINE_EXEC = globals()["PIPELINE_EXEC"]
PIPELINE_STAGES = globals()["PIPELINE_STAGES"]
EXECUTABLE_STAGES = globals()["EXECUTABLE_STAGES"]
prompt_context = globals()["prompt_context"]
run_pipeline = globals()["run_pipeline"]
run_last_failed_stage = globals()["run_last_failed_stage"]

print("Pipeline stages (run in order):")
for i, stage_name in enumerate(PIPELINE_STAGES, 1):
    stage_kind = "executes" if stage_name in EXECUTABLE_STAGES else "analysis"
    print(f"  {i}. {stage_name}  <- {stage_kind}")

print("\nRecipe deployment:")
print(f"  {SESSION_RUNTIME.goose_pipeline_recipe_path()}")
print("\nRun the full pipeline: await run_pipeline()")
print("Resume from latest saved state: await run_last_failed_stage()")

Pipeline stages (run in order):
  1. detect  <- analysis
  2. test_default_build_setup  <- executes
  3. test_default_run  <- executes
  4. install_dftracer  <- executes
  5. annotate  <- analysis
  6. build_with_dftracer  <- executes
  7. run_with_dftracer  <- executes
  8. postprocess  <- executes
  9. dfanalyzer  <- executes

Recipe deployment:
  /usr/workspace/haridev/dftracer-agents/goose/recipes/00_dftracer_pipeline.yaml

Run the full pipeline: await run_pipeline()
Resume from latest saved state: await run_last_failed_stage()


## 7. Run the Full Pipeline

Click **Run Full Pipeline** to execute all 9 stages in order through the numbered Goose YAML pipeline recipes. If a stage fails, use **Run Last Failed Stage** to reload the latest `pipeline_state.json` for the workspace and resume from the saved point.

- If a failed stage is recorded, the resume action reruns that stage.
- If no stage is failed but the pipeline still has pending stages, the resume action continues from the next pending stage through the end.
- Optionally paste the app's documentation URL before clicking. The recipe context uses it when the repository does not contain enough build guidance.
- DFTracer remains exposed through the local MCP server, while Goose recipes handle the agent and subagent orchestration.
- DFTracer is pip-installed into the workspace venv during `install_dftracer` as `dftracer[dfanalyzer]` with inferred feature flags for the app.
- Each run writes logs and state under `artifacts/run_<timestamp>/`, including per-stage attempt logs and the JSON state file used for resume and debugging flows.
- Resume runs reuse cached stage results when available, so restarting the local runtime is only necessary when a stage needs fresh planning.

In [13]:
render_pipeline_run_section()


def run_traced_command(cmd: list[str], cwd: str | pathlib.Path | None = None):
    layout = APP_STATE.get("workspace")
    if not layout:
        raise RuntimeError("Prepare the workspace first.")

    env = workspace_env(layout)
    env.update(
        {
            "DFTRACER_ENABLE": "1",
            "DFTRACER_INC_METADATA": "1",
            "DFTRACER_METADATA_USE_POSIX": "1",
            "DFTRACER_LOG_FILE": str(layout.traces / "session"),
            "DFTRACER_DATA_DIR": str(layout.repo),
        }
    )
    result = subprocess.run(cmd, cwd=str(cwd or layout.repo), env=env, text=True, capture_output=True)
    return result


print("Helper: run_traced_command([...]) — runs with DFTRACER_ENABLE=1, DFTRACER_INC_METADATA=1, DFTRACER_METADATA_USE_POSIX=1")

Stages (run in order): detect -> test_default_build_setup -> test_default_run -> install_dftracer -> annotate -> build_with_dftracer -> run_with_dftracer -> postprocess -> dfanalyzer
Each run writes stage logs plus pipeline_state.json under: <workspace>/artifacts/run_<timestamp>
Helper: run_traced_command([...]) — runs with DFTRACER_ENABLE=1, DFTRACER_INC_METADATA=1, DFTRACER_METADATA_USE_POSIX=1


## 7.5 Goose Recipe Smoke Test and Pipeline Interface

Use the cells below to verify that the notebook is talking to the numbered Goose recipe deployment directly before running the full pipeline.

This section gives you a direct interface for recipe-backed stage execution and a small smoke test that runs `detect` and `annotate` through `SESSION_RUNTIME.run_goose_pipeline_stage_recipe(...)`.

In [ ]:
import json

import ipywidgets as widgets
from IPython.display import display

pipeline_stage_widget = widgets.Dropdown(
    options=[
        ("detect", "detect"),
        ("test_default_build_setup", "test_default_build_setup"),
        ("test_default_run", "test_default_run"),
        ("annotate", "annotate"),
        ("build_with_dftracer", "build_with_dftracer"),
        ("postprocess", "postprocess"),
        ("dfanalyzer", "dfanalyzer"),
    ],
    value="detect",
    description="Stage",
)
pipeline_context_widget = widgets.Textarea(
    value="",
    placeholder="Optional extra context for the selected stage...",
    description="Context",
    layout=widgets.Layout(width="100%", height="110px"),
)
pipeline_stage_button = widgets.Button(description="Run Selected Stage", button_style="warning", icon="play")
pipeline_smoke_button = widgets.Button(description="Smoke Test Detect + Annotate", button_style="success", icon="check")
pipeline_output = widgets.Output()

def _run_recipe_stage(stage_name: str, extra_context: str = "") -> dict:
    return SESSION_RUNTIME.run_goose_pipeline_stage_recipe(stage_name, extra_context.strip())

def _on_pipeline_stage(_button) -> None:
    with pipeline_output:
        pipeline_output.clear_output()
        stage_name = pipeline_stage_widget.value
        payload = _run_recipe_stage(stage_name, pipeline_context_widget.value)
        print(json.dumps({stage_name: payload}, indent=2, sort_keys=True))

def _on_pipeline_smoke(_button) -> None:
    with pipeline_output:
        pipeline_output.clear_output()
        smoke_results = {
            "detect": _run_recipe_stage("detect", pipeline_context_widget.value),
            "annotate": _run_recipe_stage("annotate", pipeline_context_widget.value),
        }
        print(json.dumps(smoke_results, indent=2, sort_keys=True))

pipeline_stage_button.on_click(_on_pipeline_stage)
pipeline_smoke_button.on_click(_on_pipeline_smoke)

display(
    widgets.VBox(
        [
            widgets.HTML(
                "<b>Direct recipe-backed pipeline interface</b><br>"
                "Run an individual Goose recipe stage or smoke-test the detect and annotate stages before launching the full pipeline."
            ),
            pipeline_stage_widget,
            pipeline_context_widget,
            widgets.HBox([pipeline_stage_button, pipeline_smoke_button]),
            pipeline_output,
        ]
    )
)

## 8. Rerun The Last Failed Stage

Use this cell to reload the latest `pipeline_state.json` for the prepared workspace and resume the pipeline from its saved state.

- If there is a recorded failed stage, the notebook reruns that stage.
- If there is no failed stage but some later stages are still pending, the notebook continues from the next pending stage through the end.
- If the failed stage already has cached plan data, the retry can run without restarting the notebook agent. If the failure happened before the stage plan was generated, start the agent first so the stage can be planned again.

In [ ]:
from datetime import datetime

from pathlib import Path

import json



import ipywidgets as widgets

from IPython.display import display





def _latest_pipeline_state_path() -> Path:

    finder = globals().get("find_latest_pipeline_state")

    if callable(finder):

        state_path = finder()

        if state_path:

            return Path(state_path)



    layout = APP_STATE.get("workspace")

    if not layout:

        raise RuntimeError("Prepare the workspace first.")



    candidates = sorted(layout.artifacts.glob("run_*/pipeline_state.json"), key=lambda path: path.stat().st_mtime)

    if not candidates:

        raise RuntimeError("No pipeline_state.json file was found for this workspace.")

    return candidates[-1]





def revert_pipeline_state_to_annotate(state_path: Path) -> dict:

    state = json.loads(state_path.read_text(encoding="utf-8"))



    state["last_completed_stage"] = "install_dftracer"

    state["last_failed_stage"] = "annotate"

    state["next_pending_stage"] = "annotate"

    state["status"] = "partial"

    state["updated_at"] = datetime.now().isoformat(timespec="seconds")



    annotate = state["stages"].get("annotate", {})

    annotate["status"] = "failed"

    annotate["started_at"] = None

    annotate["completed_at"] = None

    annotate["latest_log"] = ""



    for stage_name in ["build_with_dftracer", "run_with_dftracer", "postprocess", "dfanalyzer"]:

        stage = state["stages"].get(stage_name)

        if not stage:

            continue

        stage["status"] = "pending"

        stage["started_at"] = None

        stage["completed_at"] = None

        stage["latest_log"] = ""



    state_path.write_text(json.dumps(state, indent=2), encoding="utf-8")

    APP_STATE["last_pipeline_state_file"] = str(state_path)

    return state





revert_state_output = widgets.Output()

revert_state_button = widgets.Button(

    description="Reset Latest Run To Annotate",

    button_style="warning",

    tooltip="Marks annotate as the failed stage and resets later stages to pending.",

    icon="history",

)





def _handle_revert_state(_button) -> None:

    with revert_state_output:

        revert_state_output.clear_output()

        try:

            state_path = _latest_pipeline_state_path()

            state = revert_pipeline_state_to_annotate(state_path)

            print(

                json.dumps(

                    {

                        "pipeline_state": str(state_path),

                        "last_completed_stage": state.get("last_completed_stage"),

                        "last_failed_stage": state.get("last_failed_stage"),

                        "next_pending_stage": state.get("next_pending_stage"),

                    },

                    indent=2,

                )

            )

        except Exception as exc:

            print(f"Failed to reset pipeline state: {exc}")





revert_state_button.on_click(_handle_revert_state)



display(

    widgets.VBox(

        [

            widgets.HTML(

                "<b>Reset the latest saved run to the annotate stage</b><br>"

                "Use this before rerunning the pipeline when you want stage 5 onward to execute again."

            ),

            revert_state_button,

            revert_state_output,

        ]

    )

)


In [ ]:
from dftracer_agents.notebook.pipeline import install_notebook_pipeline

PIPELINE_RUNTIME = install_notebook_pipeline(globals())
run_last_failed_stage = globals()["run_last_failed_stage"]

await run_last_failed_stage()

## 8. Review Trace Outputs and Pipeline Results

This section summarizes saved answers and any trace files found in the workspace.

In [ ]:
layout = APP_STATE.get("workspace")

if not layout:

    raise RuntimeError("Prepare the workspace first.")



print("Pipeline results summary:")

for stage, value in PIPELINE_RESULTS.items():

    print(f"\n[{stage}]")

    print(str(value)[:800])



current_trace_dir = APP_STATE.get("current_trace_dir")

if current_trace_dir:

    trace_root = pathlib.Path(str(current_trace_dir))

else:

    run_id = APP_STATE.get("current_run_id")

    trace_root = layout.traces / str(run_id) if run_id else layout.traces



trace_files = list(trace_root.rglob("*.pfw")) + list(trace_root.rglob("*.pfw.gz")) if trace_root.exists() else []

print(f"\nTrace root: {trace_root}")

print(f"Trace files discovered: {len(trace_files)}")

for path in trace_files[:20]:

    print(f"  {path}")



artifact_file = layout.artifacts / "pipeline_results.json"

artifact_file.write_text(json.dumps(PIPELINE_RESULTS, indent=2, default=str))

print(f"\nSaved results to {artifact_file}")


## 9. Ask Questions and Give Feedback in the Notebook

Use the options and free-text box below to keep the workflow conversational.

Preset prompts are provided to make the interaction easier.

In [ ]:
render_chat_section()


In [ ]:
render_outcome_feedback_section()


## 10. Save Session Diagnostics

This writes the current notebook session state into the workspace `artifacts/` directory for reuse.

In [ ]:
layout = APP_STATE.get("workspace")
if not layout:
    raise RuntimeError("Prepare the workspace first.")

session_file = layout.artifacts / "notebook_session.json"
payload = {
    "repo_url": APP_STATE.get("repo_url"),
    "branch": APP_STATE.get("branch"),
    "workspace": layout.as_dict(),
    "repo_attrs": APP_STATE.get("repo_attrs"),
    "feedback": APP_STATE.get("feedback"),
    "results": APP_STATE.get("results"),
    "log_count": len(APP_STATE.get("logs", [])),
}
session_file.write_text(json.dumps(payload, indent=2, default=str))
print(f"✓ Saved session to {session_file}")
print("\nSaved keys:")
for key in payload:
    print(f"  - {key}")

In [ ]:
from pathlib import Path
import importlib

import dftracer_agents.mcp_servers.modules.annotations as ann
import dftracer_agents.mcp_servers.modules.dftracer as dft

ann = importlib.reload(ann)
dft = importlib.reload(dft)

repo_dir = PROJECT_ROOT / "workspaces" / "ior" / "source" / "ior"
result = dft.auto_annotate_application(
    repo_dir=str(repo_dir),
    language="c",
    max_files=200,
    patch_build_files=False,
    dry_run=False,
)
print({"modified_files": result["modified_files"]})

ior_path = repo_dir / "src" / "ior.c"
ior_text = ann.safe_read_text(ior_path)
ior_spans = ann.collect_c_functions_with_llvm(ior_path, ior_text)
missing = [
    span.name
    for span in ior_spans
    if "DFTRACER_C_FUNCTION_START();" not in ior_text[span.open_brace + 1 : span.close_brace]
]
print({"ior_total_functions": len(ior_spans), "ior_missing_annotations": missing})